# Notebook 60R2

Generates `p_print` directly from 59P and feeds it to 59R1. No pre-generated `p_print` files.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy scikit-learn joblib torch torchvision pyarrow

import json,hashlib,joblib,cv2,torch,numpy as np,pandas as pd
from pathlib import Path
from torch import nn
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
OP_RUN=Path('SET_59P_RUN_ROOT')
RM_RUN=Path('SET_59R1_RUN_ROOT')
REVIEW_AUDIT_DIR=ROOT/'human_in_the_middle'/'audit'
OUT=ROOT/'notebook60r2_integrated_runtime'
OUT.mkdir(parents=True,exist_ok=True)
def req(x,m):
 if not x: raise RuntimeError(m)
def sha(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def js(p): return json.loads(Path(p).read_text())
# Exact sibling replay contract
OB=OP_RUN/'bundle'; OR=OP_RUN/'replay'/'replay_test.json'
RB=RM_RUN/'bundle'; RR=RM_RUN/'replay'/'replay_test.json'
for p in [OB/'observed_print_model.pt',OB/'observed_print_calibrator.joblib',OB/'channel_schema.json',OB/'policy.json',OB/'freeze_manifest.json',OR,RB/'remove_model.joblib',RB/'remove_calibrator.joblib',RB/'predictor_schema.json',RB/'policy.json',RB/'freeze_manifest.json',RR]: req(p.is_file(),f'Missing {p}')
opr,rmr=js(OR),js(RR);req(opr.get('passed') is True and rmr.get('passed') is True,'Replay failure')
ops,opp=js(OB/'channel_schema.json'),js(OB/'policy.json');rms,rmp=js(RB/'predictor_schema.json'),js(RB/'policy.json')
CH=ops.get('channels') or ops.get('ordered_channels');PRED=rms.get('ordered_predictors') or rms.get('predictors')
req(CH==['obs','norm','mask','depth','horizontal','vertical','compact','preserve','remove','ambiguous','edge'],'59P schema mismatch');req(len(PRED)==17,'59R1 schema mismatch')
class TopologyCNN(nn.Module):
 def __init__(self):
  super().__init__();self.net=nn.Sequential(nn.Conv2d(11,24,3,padding=1),nn.ReLU(),nn.BatchNorm2d(24),nn.MaxPool2d(2),nn.Conv2d(24,48,3,padding=1),nn.ReLU(),nn.BatchNorm2d(48),nn.MaxPool2d(2),nn.Conv2d(48,64,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1));self.head=nn.Linear(64,2)
 def forward(self,x):return self.head(self.net(x).flatten(1))
device='cuda' if torch.cuda.is_available() else 'cpu';op_model=TopologyCNN().to(device);state=torch.load(OB/'observed_print_model.pt',map_location=device);state=state.get('state_dict',state);state={str(k).removeprefix('module.'):v for k,v in state.items()};op_model.load_state_dict(state);op_model.eval();op_cal=joblib.load(OB/'observed_print_calibrator.joblib');rm_model=joblib.load(RB/'remove_model.joblib');rm_cal=joblib.load(RB/'remove_calibrator.joblib')

def prepare(obs,seed):
 mask=cv2.dilate(np.where(seed>0,255,0).astype(np.uint8),cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5)));vis=(mask==0)&(obs>=200);num=cv2.GaussianBlur(obs.astype(np.float32)*vis,(0,0),45);den=cv2.GaussianBlur(vis.astype(np.float32),(0,0),45);paper=np.clip(np.where(den>.02,num/np.maximum(den,1e-4),cv2.GaussianBlur(obs,(0,0),45)),0,255).astype(np.uint8);norm=cv2.divide(obs,np.maximum(paper,1),scale=255);dark=(obs<190)&(mask==0);h=cv2.dilate((cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0).astype(np.uint8),np.ones((9,9),np.uint8))>0;v=cv2.dilate((cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((15,1),np.uint8))>0).astype(np.uint8),np.ones((9,9),np.uint8))>0;n,l,s,_=cv2.connectedComponentsWithStats(dark.astype(np.uint8),8);compact=np.zeros_like(dark)
 for i in range(1,n):
  x,y,w,hg,a=s[i]
  if 3<=a<=650 and w<=40 and hg<=40:compact[l==i]=1
 compact=cv2.dilate(compact.astype(np.uint8),np.ones((9,9),np.uint8))>0;inside=mask>0;preserve=inside&(h|v|compact)&(obs<200);remove=inside&~preserve&(obs>=80)&(norm>=80);ambiguous=inside&~preserve&~remove;telea=cv2.inpaint(obs,mask,3,cv2.INPAINT_TELEA);v78=obs.copy();v78[remove]=paper[remove];v78[ambiguous]=telea[ambiguous]
 return mask,v78,{'obs':obs,'norm':norm,'mask':inside,'depth':cv2.distanceTransform(inside.astype(np.uint8),cv2.DIST_L2,5),'horizontal':h,'vertical':v,'compact':compact,'preserve':preserve,'remove':remove,'ambiguous':ambiguous,'edge':cv2.Canny(obs,70,170)>0,'paper':paper}
def make_p_print(a):
 elig=a['mask']&(a['obs']<int(opp.get('dark_threshold',200)))&~a['preserve'];ys,xs=np.where(elig);p=np.full(a['obs'].shape,np.nan,np.float32);pad={k:np.pad((a[k].astype(np.float32)/255 if k in ['obs','norm'] else a[k].astype(np.float32)),16,mode='reflect') for k in CH}
 with torch.no_grad():
  for st in range(0,len(ys),2048):
   yy,xx=ys[st:st+2048],xs[st:st+2048];z=np.stack([np.stack([pad[k][y:y+33,x:x+33] for k in CH]) for y,x in zip(yy,xx)]);raw=torch.softmax(op_model(torch.from_numpy(z).to(device)),1)[:,1].cpu().numpy();p[yy,xx]=op_cal.predict(raw)
 return p,elig
# Resolve reviewed pages and run. 59R1 predictor names are authoritative: unknown names stop.
rows=[]
for ap in sorted(REVIEW_AUDIT_DIR.glob('*.json')):
 v=js(ap);op=Path(v.get('aligned_output',''));mp=Path(v.get('corrected_mask_output',''))
 if op.is_file() and mp.is_file():rows.append((str(v.get('page') or op.stem),op,mp))
req(rows,'No Notebook07 reviewed pages')
summary=[]
for page,op,mp in rows:
 obs=cv2.imread(str(op),0);seed=cv2.imread(str(mp),0);req(obs is not None and seed is not None and obs.shape==seed.shape,page+' input failure');m3,v78,a=prepare(obs,seed);pprint,elig=make_p_print(a);y,x=np.where(m3>0);base={'observed_value':obs[y,x],'normalised_value':a['norm'][y,x],'paper_value':a['paper'][y,x],'mask_depth':a['depth'][y,x],'edge':a['edge'][y,x].astype(float),'p_print':np.nan_to_num(pprint[y,x],nan=0.0)}
 # Predictor contract is fail-closed. Add aliases here only if 59R1 schema names differ.
 missing=[k for k in PRED if k not in base];req(not missing,f'{page}: implement exact 59R1 predictors: {missing}')
 X=pd.DataFrame(base)[PRED];pr=rm_cal.predict(rm_model.predict_proba(X)[:,1]);safe=np.zeros(obs.shape,bool);safe[y[pr>=float(rmp['threshold'])],x[pr>=float(rmp['threshold'])]]=1;m5=(m3>0)&~safe;corrected=cv2.inpaint(obs,(m5*255).astype(np.uint8),3,cv2.INPAINT_TELEA);corrected[m3==0]=obs[m3==0];d=OUT/page;d.mkdir(parents=True,exist_ok=True);cv2.imwrite(str(d/'corrected.png'),corrected);np.savez_compressed(d/'audit.npz',p_print=pprint,p_remove=pr,y=y,x=x,safe_remove=safe,m3=m3>0,m5_remove=m5);summary.append({'page':page,'p_print_eligible':int(elig.sum()),'safe_remove':int(safe.sum()),'m5_remove':int(m5.sum()),'outside_drift':int((corrected[m3==0]!=obs[m3==0]).sum())})
pd.DataFrame(summary).to_csv(OUT/'page_summary.csv',index=False)
contract={'pre_generated_p_print_used':False,'observed_print_policy_id':opp.get('policy_id'),'observed_print_compatibility_id':opp.get('compatibility_id'),'remove_policy_id':rmp.get('policy_id'),'remove_compatibility_id':rmp.get('compatibility_id'),'59P_replay':str(OR),'59R1_replay':str(RR),'pages':len(summary),'production_authority':False};(OUT/'combined_runtime_contract.json').write_text(json.dumps(contract,indent=2));print(contract)
